In [27]:
from agents import Agent, WebSearchTool, FileSearchTool , function_tool,Runner
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
import asyncio
import os
from typing import Dict
from dotenv import load_dotenv
from openai import AsyncOpenAI
import requests
from agents import RunConfig
from agents import set_default_openai_client
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel

In [28]:
load_dotenv(override=True)

True

In [29]:
from agents import set_tracing_disabled

set_tracing_disabled(True)

In [30]:
pushover_token=os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"
pushover_user= os.getenv('PUSHOVER_USER')

In [31]:
def push(message):
    payload={"user":pushover_user, "token":pushover_token,"message":message}
    requests.post(pushover_url,data=payload)

In [32]:
from openai import base_url


client = AsyncOpenAI(
    api_key = os.getenv('DEEPSEEK_API_KEY'),
    base_url = "https://api.deepseek.com"
)

set_default_openai_client(client)

In [33]:
orchestrator_instructions = """You are the orchestrator.

Workflow:
1. Call worker exactly once for the user task.
2. Call eval exactly once to review the worker output.
3. Return the final answer to the user.
Do not call tools repeatedly.
Do not delegate again after evaluation."""



evaluator_instructions = """You are the Evaluator Agent.

Your role is to critically review outputs generated by Worker Agents.

Responsibilities:

Check factual correctness
Detect hallucinations or unsupported claims
Verify completeness relative to the assigned task
Ensure formatting and structure are correct
Identify logical inconsistencies
Suggest improvements or corrections if needed

Evaluation Criteria:

Accuracy
Relevance
Completeness
Clarity
Reliability

Rules:

Be strict and analytical.
Never assume outputs are correct.
If output quality is poor, explain exactly what should be fixed.
If output is acceptable, explicitly approve it.

Output Format:

Verdict: PASS / FAIL
Issues Found
Suggested Fixes
Confidence Score (1-10) """

worker_instruction = """ You are a Worker Agent specialized in executing assigned tasks.

Your responsibilities:

Focus only on the assigned subtask
Produce accurate and concise outputs
Avoid unnecessary explanations
Return structured and actionable results
Ask for clarification only if critical information is missing

Rules:

Do not change the task objective.
Do not perform unrelated actions.
Optimize for correctness and efficiency.
Return outputs in a machine-readable format whenever possible.
If the task cannot be completed, explain why clearly.

Behavior:

Execute first
Minimize reasoning verbosity
Stay task-focused
Be reliable and deterministic

Output Format:

Task Summary
Execution Result
Errors/Limitations (if any)"""


In [34]:


evaluator_agent = Agent(
    name='evaluator',
    instructions = evaluator_instructions,
    model=OpenAIChatCompletionsModel(
        model='deepseek-v4-flash',
        openai_client=client
    )
)

worker_agent=Agent(
    name='worker',
    instructions = worker_instruction,
    model=OpenAIChatCompletionsModel(
        model='deepseek-v4-flash',
        openai_client=client
    )
    
)



In [35]:

evaluator_tool = evaluator_agent.as_tool(tool_name='eval', tool_description='you are an evaluator.')
worker_tool = worker_agent.as_tool(tool_name='worker', tool_description='you are a worker whose job is to peform all tasks provided to you')


In [36]:
get_tools=[worker_tool,evaluator_tool]

In [37]:
orchestrator_agent = Agent(
    name='orchestrator',
    instructions = orchestrator_instructions,
    model = OpenAIChatCompletionsModel(
        model = 'deepseek-v4-flash',
        openai_client = client
    ),
    tools = get_tools
)
print("got tools")

got tools


In [38]:
type(get_tools)
print(get_tools)

[FunctionTool(name='worker', description='you are a worker whose job is to peform all tasks provided to you', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7ae2bc871d90>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False), FunctionTool(name='eval', description='you are an evaluator.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invok

In [39]:
result = await Runner.run(
    orchestrator_agent,
    "Tell me about max verstappen",
    max_turns=3
)
print(result.final_output)
push(result.final_output)

# Max Verstappen 🏎️

Max Verstappen is a **Dutch Formula 1 driver**, born **September 30, 1997**, who races for **Red Bull Racing**. He is widely regarded as one of the greatest talents in F1 history.

---

## 🏆 Championships
- **4-time World Drivers' Champion** (2021, 2022, 2023, 2024)

## 🚀 Career Journey
- Debuted in F1 in **2015** with Scuderia Toro Rosso at just **17 years old** — becoming the **youngest driver in F1 history**
- Moved to Red Bull in **2016** and won his **first race on debut** at the Spanish Grand Prix
- Has since become the dominant force in modern F1

## 🔥 Major Achievements & Records
- **Most wins in a single season**: 19 (2023)
- **Most consecutive wins**: 10 (2023)
- **Youngest Grand Prix winner ever** (18 years, 228 days)
- **2023 season**: Broke records for most wins, most podium finishes (21), and most points in a season
- **2024**: Secured his fourth consecutive world title

## 💡 Key Facts
- Son of former F1 driver **Jos Verstappen** and karting champion 